# 01｜IMDB v1.1 去重版文字清理與向量化

Train/Validation/Test 使用同一份已核定的清理規則和切分檔案。 BoW、TF-IDF、Keras TextVectorization **全部重新從 v1.1 Train fit/adapt**，絕不重用 v1.0 的詞彙表或 Token IDs。

保留否定詞及標點，TextVectorization 使用 12,000 詞彙、長度 160。

In [1]:
from pathlib import Path
import json
import sys
ROOT=Path('/workspace/imdb_matrix_experiment/variants/dedup_v1_1')
assert ROOT.exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from src.common import config,load_splits,clean_review
from src.feature_utils import prepare_features
import json
import pandas as pd
import numpy as np
c,c_hash=config()
manifest=json.loads((ROOT/"data/processed/split_manifest.json").read_text())
assert manifest["config_sha256"]==c_hash
print("Variant",c["version"],"preprocessing",c["preprocessing"]["version"])
print("Canonical split SHA",manifest["split_assignments_sha256"])


Variant dedup_v1.1 preprocessing phase0-text-v1
Canonical split SHA 9e25b0d714d1fb555c8c8a58fa6858c392ec60ecfba554b923553626154174e9


## 1. 相同清理函式對照
HTML entity → <br> 空白 → 其他 HTML 標籤去除 → 小寫 → 空白規範化；標點和 stopwords 保留。

In [2]:
df=load_splits()["train"]
for i in range(2):
    original=df.iloc[i].text
    cleaned=clean_review(original)
    assert cleaned==df.iloc[i].cleaned_text
    print("RAW:",original[:190].replace("\n"," "))
    print("CLEAN:",cleaned[:190])
    print("---")


RAW: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. custom
CLEAN: i rented i am curious-yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by u.s. custom
---
RAW: If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because i
CLEAN: if only to avoid making this type of film in the future. this film is interesting as an experiment but tells no cogent story. one might feel virtuous for sitting thru it because it touches o
---


## 2. v1.1 Train-only 擬合與 Validation/Test 轉換
只用新的 19,823 筆 Train 學習詞彙表及 IDF；Validation/Test 僅 transform/inference。另保存六個稀疏矩陣供 Phase 1 使用。

In [3]:
features=prepare_features()
print("Fitted on:",features["fitted_on"])
print("Feature dimensions:",features["features"])
print("Split SHA:",features["manifest_split_sha256"])
print("Fit policy:",features["fit_policy"])


I0000 00:00:1793791099.127395    4277 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4599 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:02:00.0, compute capability: 7.5


Fitted on: dedup_v1.1 train only
Feature dimensions: {'bow': 12000, 'tfidf': 12000, 'keras': 12000}
Split SHA: 9e25b0d714d1fb555c8c8a58fa6858c392ec60ecfba554b923553626154174e9
Fit policy: {'bow': 'fit_transform(train)', 'tfidf': 'fit_transform(train)', 'keras': 'adapt(train)'}


## 3. Keras Token ID 和處理結果檢查
ID=0 代表 Padding，ID=1 代表未知詞 OOV；所有輸入固定 160 tokens。

In [4]:
art=ROOT/"artifacts/vectorizers"
vocab=(art/"keras_vocabulary_12000.txt").read_text().splitlines()
assert vocab[0]=="" and vocab[1]=="[UNK]"
with np.load(art/"keras_token_ids_12000_160.npz",allow_pickle=False) as x:
    for name in ("train","validation","test"):
        print(name,x[f"X_{name}"].shape,x[f"y_{name}"].shape)
    print("First 20 token IDs:",x["X_train"][0,:20].tolist())
print("Vocab first 15:",vocab[:15])


train (19823, 160) (19823,)
validation (4956, 160) (4956,)
test (25000, 160) (25000,)
First 20 token IDs: [9, 1454, 9, 224, 1, 34, 52, 423, 1409, 74, 5, 33, 2, 10156, 11, 3361, 12, 49, 12, 13]
Vocab first 15: ['', '[UNK]', 'the', 'a', 'and', 'of', 'to', 'is', 'in', 'i', 'this', 'that', 'it', 'was', 'as']


## 4. 長評論截斷／OOV 分析與安全限制
原規格維持 max_len=160；在正式報告列出截斷比例。 Gate A 專注於完全相同清理後全文；近似語意仍是限制。

In [5]:
print(pd.DataFrame(features["lengths"]).T.to_string())
print("Feature metadata:", ROOT/"results/feature_metadata.json")
print("Note: Deduplicated v1.1 features cannot be mixed with v1.0 models.")


               rows  mean_token_count  truncated_records  truncated_fraction  oov_token_fraction_before_truncation
train       19823.0           231.098            10941.0            0.551935                              0.108112
validation   4956.0           234.105             2743.0            0.553471                              0.110814
test        25000.0           226.250            13583.0            0.543320                              0.112527
Feature metadata: /workspace/imdb_matrix_experiment/variants/dedup_v1_1/results/feature_metadata.json
Note: Deduplicated v1.1 features cannot be mixed with v1.0 models.
